# Задание #3. Фильтрация ложных данных

Система [ATLAS](https://en.wikipedia.org/wiki/Asteroid_Terrestrial-impact_Last_Alert_System) каждую ночь сканирует небо в поиске движущихся объектов: астероидов, комет и транзиентных космических явлений (временные явления в жизненном цикле звезд). Для поиска изменений используется в том числе метод разностной фотометрии: из нового снимка вычитается старый эталон.

Однако автоматика выдает тысячи ложных срабатываний из-за бликов линз, дефектов матрицы и цифрового шума. Если астрономы будут проверять каждое такое срабатывание вручную, они могут пропустить реальную угрозу. Ваша миссия — создать высокоточный классификатор, который станет фильтром для телескопа и отсеет цифровой мусор, оставив только настоящие космические явления.

## Данные
Имеется набор из `1,998` изображений, взятых из открытых наборов [ATLAS](https://atlas.fallingstar.com/). Изображения представляют собой входные данные для анализа космических транзиентов. Каждое изображение это триплет из:
- новый снимок (`New`);
- существующий эталон (`Ref`);
- результат вычета (`Dif`).

Пример триплета:
</br></br>
![transient](https://i.imgur.com/cOx8Gmd.png)

### Структура данных
Дано два поля:
- `image`: композиция для анализа транзиента `PIL.Image.Image`, исходный размер — `950x362` пикселя.
- `label`: бинарное значение (0 или 1), определяющее, является ли изображение подленным

Набор делится на:
- тренировочные данные: `~1.8k` изображений
- тестовые данные: `200` изображений


### Загрузка данных
Обратитесь к администрации и уточните, где можно загрузить набор, если нет доступа к ресурсу HuggingFace.

In [ ]:
import polars as pl

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pl.read_parquet("hf://datasets/Santhosh2312/ATLAS/" + splits["train"])

In [ ]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Santhosh2312/ATLAS/" + splits["train"])

In [ ]:
from datasets import load_dataset

ds = load_dataset("Santhosh2312/ATLAS")

## Метрики
В данной задаче цена ошибки неодинакова: пропустить реальный астероид гораздо опаснее, чем заставить астронома лишний раз взглянуть на шум. Поэтому при оценивании будет отдан приоритет **полноте**. Исходя из этого возьмем за основную метрику $ F_{\beta} $, поставим приоритет `recall` над `precision` равным $\beta = 2$: 
</br></br>
$$
F_{\beta} = \frac{(1 + \beta^{2}) \cdot TP}{(TP + FN) \cdot \beta^2 + (TP + FP)}
$$
</br></br>
Вспомогательной метрикой будем считать стандартный `ROC AUC` для оценки способности модели разделять классы при различных порогах вероятности.
</br></br>
Кроме этого будет учитываться **время обработки** одного изображения и **конечный вес модели**.

# Ограничение
Запрещено:
- дообучать моделы на набораx данных кроме представленного;
- обучать модели на тестовых наборах.

Модель обязана анализировать контекст (все три кадра `New`, `Ref`, `Dif`).  Работы, которые нарушили данные ограничения, **рассмотрены не будут**.

# Baseline
Ниже приведен базовый стенд для тестирования модели, обученной с помощью `PyTorch`

In [ ]:
import os
import time
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from sklearn.metrics import fbeta_score, roc_auc_score

class AtlasTransientValidator:
    def __init__(self, model_path, test_limit=None):
        self.model_path = model_path
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.test_limit = test_limit
        
        self.target_size = (224, 672)
        
        self.transform = transforms.Compose([
            transforms.Resize(self.target_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def load_test_data(self):
        ds = load_dataset("Santhosh2312/ATLAS", split="test")
        if self.test_limit:
            ds = ds.select(range(self.test_limit))
        return ds

    def evaluate(self, model):
        test_ds = self.load_test_data()
        all_probs = []
        all_preds = []
        all_labels = []
        latencies = []

        model.eval()
        model.to(self.device)
        
        with torch.no_grad():
            for item in test_ds:
                image = item['image'].convert("RGB")
                label = item['label']
                
                input_tensor = self.transform(image).unsqueeze(0).to(self.device)
                
                start_time = time.perf_counter()
                output = model(input_tensor)
                prob = torch.softmax(output, dim=1)[:, 1].item() if output.shape[1] > 1 else torch.sigmoid(output).item()
                end_time = time.perf_counter()
                
                all_probs.append(prob)
                all_preds.append(1 if prob > 0.5 else 0)
                all_labels.append(label)
                latencies.append(end_time - start_time)

        results = {
            "f2_score": fbeta_score(all_labels, all_preds, beta=2),
            "roc_auc": roc_auc_score(all_labels, all_probs),
            "avg_latency_ms": np.mean(latencies) * 1000,
            "model_size_mb": os.path.getsize(self.model_path) / (1024 * 1024)
        }
        
        return results

    def run_check(self, model_instance):
        try:
            state_dict = torch.load(self.model_path, map_location=self.device)
            model_instance.load_state_dict(state_dict)
            
            results = self.evaluate(model_instance)
            
            print("\n" + "="*40)
            print("ATLAS SYSTEM: SCAN REPORT")
            print("="*40)
            print(f"F2-Score (Priority Recall): {results['f2_score']:.4f} (ОСНОВНАЯ)")
            print(f"ROC AUC:                   {results['roc_auc']:.4f}")
            print(f"Avg Inference Time:        {results['avg_latency_ms']:.2f} ms")
            print(f"Model Weight:              {results['model_size_mb']:.2f} MB")
            print("="*40)
            
            if results['model_size_mb'] > 30:
                print("WARNING: Model exceeds 40MB limit!")

        except Exception as e:
            print(f"Ошибка валидации ATLAS: {e}")